# Final Comparison: XLM-RoBERTa vs MuRIL vs mBERT
## All 3 Models × 3 Test Sets — Paper Table Generator

This notebook loads results saved by the 3 model notebooks
and produces the final comparison tables and figures for your paper.

### What you get:
- **Table 1** — Main results: all 3 models × all 3 test sets (Macro F1)
- **Table 2** — XAI faithfulness (AOPC) comparison across models and test sets
- **Figure 1** — Grouped bar chart: model comparison across test sets
- **Figure 2** — AOPC heatmap: XAI faithfulness visualization
- **Figure 3** — All confusion matrices (3×3 grid)


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path

XLMR_DIR  = "/content/drive/MyDrive/xlmr_results"
MURIL_DIR = "/content/drive/MyDrive/muril_results"
MBERT_DIR = "/content/drive/MyDrive/mbert_results"
OUT_DIR   = "/content/drive/MyDrive/comparison_results"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Setup done")

## Load Results from All 3 Models

In [ ]:
import os

def load_results(model_dir, model_name):
    path = f"{model_dir}/{model_name.lower().replace('-','').replace(' ','_')}_results.csv"
    # Try common naming
    for fname in ['xlmr_results.csv','muril_results.csv','mbert_results.csv','results.csv']:
        fpath = f"{model_dir}/{fname}"
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df['model'] = model_name
            return df
    print(f"  WARNING: No results file found in {model_dir}")
    return None

xlmr_df  = load_results(XLMR_DIR,  "XLM-RoBERTa")
muril_df = load_results(MURIL_DIR, "MuRIL")
mbert_df = load_results(MBERT_DIR, "mBERT")

# Combine (only models that finished)
dfs = [d for d in [xlmr_df, muril_df, mbert_df] if d is not None]
all_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if not all_df.empty:
    print("\nCombined results loaded:")
    print(all_df[['model','dataset','macro_f1','weighted_f1']].to_string(index=False))
else:
    print("No results found — run model notebooks first OR use demo below")

## Demo Mode — Simulate Results for Testing

If you haven't run all 3 model notebooks yet, use this demo data to test the comparison pipeline.

In [ ]:
# DEMO — replace with real results after running model notebooks
demo_results = {
    'model':    ['XLM-RoBERTa']*3 + ['MuRIL']*3 + ['mBERT']*3,
    'dataset':  ['Hindi','English','Combined'] * 3,
    'macro_f1': [0.782, 0.741, 0.763,   # XLM-RoBERTa
                  0.801, 0.698, 0.754,   # MuRIL (better on Hindi)
                  0.694, 0.712, 0.703],  # mBERT (lower bound)
    'weighted_f1': [0.791, 0.748, 0.771, 0.808, 0.705, 0.762, 0.701, 0.719, 0.710],
    'precision':   [0.785, 0.745, 0.767, 0.805, 0.700, 0.758, 0.698, 0.715, 0.706],
    'recall':      [0.779, 0.737, 0.759, 0.797, 0.696, 0.750, 0.690, 0.709, 0.700],
}
all_df = pd.DataFrame(demo_results)

# XAI demo results
xai_results = {
    'model':     ['XLM-RoBERTa']*3 + ['MuRIL']*3 + ['mBERT']*3,
    'dataset':   ['Hindi','English','Combined'] * 3,
    'aopc_raw':  [0.042, 0.038, 0.040, 0.045, 0.035, 0.041, 0.031, 0.029, 0.030],
    'aopc_norm': [0.061, 0.055, 0.058, 0.063, 0.051, 0.059, 0.044, 0.042, 0.043],
    'delta':     [0.019, 0.017, 0.018, 0.018, 0.016, 0.018, 0.013, 0.013, 0.013],
}
xai_df = pd.DataFrame(xai_results)

print("Demo data loaded. Replace with real results when available.")
print(all_df[['model','dataset','macro_f1']].to_string(index=False))

## TABLE 1 — Main Results

In [ ]:
pivot = all_df.pivot(index='model', columns='dataset', values='macro_f1')
# Reorder columns
cols = [c for c in ['Hindi','English','Combined'] if c in pivot.columns]
pivot = pivot[cols]
pivot['Average'] = pivot.mean(axis=1)

print("\n" + "="*65)
print("  TABLE 1: Macro F1 — All Models × All Test Sets")
print("="*65)
print(pivot.round(4).to_string())
print("="*65)
pivot.to_csv(f"{OUT_DIR}/table1_main_results.csv")
print("  Saved → table1_main_results.csv")

## TABLE 2 — XAI Faithfulness

In [ ]:
xai_pivot = xai_df.pivot_table(index='model', columns='dataset',
                               values=['aopc_raw','aopc_norm','delta'])
print("\nTABLE 2: XAI Faithfulness (AOPC)")
print(xai_pivot.round(4).to_string())
xai_df.to_csv(f"{OUT_DIR}/table2_xai_results.csv", index=False)

## FIGURE 1 — Grouped Bar Chart: All Models × Test Sets

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
datasets = ['Hindi', 'English', 'Combined']
models   = ['XLM-RoBERTa', 'MuRIL', 'mBERT']
colors   = ['#185FA5', '#0F6E56', '#BA7517']
x        = np.arange(len(datasets))
width    = 0.25

for i, (model, color) in enumerate(zip(models, colors)):
    vals = []
    for ds in datasets:
        row = all_df[(all_df['model']==model) & (all_df['dataset']==ds)]
        vals.append(row['macro_f1'].values[0] if len(row) else 0)
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width, label=model, color=color, alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+.005,
                f'{v:.3f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xlabel('Test Dataset', fontsize=12)
ax.set_ylabel('Macro F1 Score', fontsize=12)
ax.set_title('Cross-Lingual Transfer: Hinglish-Trained Models on 3 Test Sets', fontsize=13)
ax.set_xticks(x); ax.set_xticklabels(datasets, fontsize=11)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11, loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig1_model_comparison.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved")

## FIGURE 2 — AOPC Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes,
        ['aopc_raw', 'aopc_norm'],
        ['AOPC — Raw Input', 'AOPC — Normalized Input']):
    heat = xai_df.pivot(index='model', columns='dataset', values=metric)
    sns.heatmap(heat, annot=True, fmt='.4f', cmap='Blues',
                ax=ax, vmin=0, vmax=0.08,
                xticklabels=heat.columns, yticklabels=heat.index)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Test Dataset'); ax.set_ylabel('Model')
plt.suptitle('XAI Faithfulness (AOPC): Raw vs Normalized Input', fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig2_aopc_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved")

## FIGURE 3 — Delta AOPC (Normalization Improvement)

In [ ]:
delta_pivot = xai_df.pivot(index='model', columns='dataset', values='delta')
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(delta_pivot, annot=True, fmt='.4f', cmap='Greens',
            ax=ax, vmin=0, vmax=0.025)
ax.set_title('AOPC Improvement from Normalization (Δ = Normalized − Raw)', fontsize=12)
ax.set_xlabel('Test Dataset'); ax.set_ylabel('Model')
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fig3_aopc_delta.png", dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved")

## Paper Summary Output

In [ ]:
print("="*70)
print("  PAPER RESULTS SUMMARY")
print("="*70)
print("\nTABLE 1 — Macro F1 (Train:Hinglish, Test: 3 sets)")
print(pivot.round(4).to_string())
print("\nTABLE 2 — XAI Faithfulness (AOPC)")
print(xai_df[['model','dataset','aopc_raw','aopc_norm','delta']].to_string(index=False))
print("\nAll figures saved to:", OUT_DIR)
print("="*70)
print("\nPaper writing guide:")
print("  Table 1 → Section 5 (Results)")
print("  Table 2 → Section 6 (XAI Analysis)")
print("  Fig 1   → Section 5 Figure 1")
print("  Fig 2   → Section 6 Figure 2")
print("  RQ1: Does normalization augmentation improve F1? Compare E2 vs E4")
print("  RQ2: Does Hinglish model transfer to Hindi better than English model?")
print("  RQ3: Does normalization improve XAI faithfulness (AOPC)?")